<a href="https://colab.research.google.com/github/gunamn06/NLP/blob/main/minimal_seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim


# Dummy data:
input_seq = torch.tensor([[1, 2, 3]], dtype=torch.long)   # (batch, seq_len)
target_seq = torch.tensor([[3, 2, 1]], dtype=torch.long)

vocab_size = 10 # toy vocabulary size
embedding_dim = 8
hidden_size = 16



class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)         # (batch, seq_len, embed_dim)
        outputs, hidden = self.rnn(embedded) # outputs: (batch, seq_len, hidden), hidden: (layers, batch, hidden)
        return hidden

In [2]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, target, hidden):
        embedded = self.embedding(target)      # (batch, seq_len, embed_dim)
        outputs, _ = self.rnn(embedded, hidden)# outputs: (batch, seq_len, hidden), hidden: (1, batch, hidden)
        logits = self.fc(outputs)              # (batch, seq_len, vocab_size)
        return logits



encoder = Encoder(vocab_size, embedding_dim, hidden_size)
decoder = Decoder(vocab_size, embedding_dim, hidden_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)

In [4]:
encoder_hidden = encoder(input_seq)  # hidden: (1, batch, hidden)
logits = decoder(target_seq, encoder_hidden)


logits.shape # (batch, seq_len, vocab_size)

torch.Size([1, 3, 10])

In [5]:
# One epoch training
encoder_hidden = encoder(input_seq)  # hidden: (1, batch, hidden)
logits = decoder(target_seq, encoder_hidden)
